## Knowledge Distillation

Knowledge distillation (KD) is widely used for compressing a teacher model to reduce its inference cost and memory footprint, by training a smaller student model. Auto-regressive sequence models, such as language models (LMs), have shown impressive capabilities in numerous tasks, where the key to this success is often scaling the amount of training data as well as the number of model parameters. However, scaling parameter count comes at a cost, and the deployment of such models is limited by either their inference cost or memory footprint. Thus, a crucial goal for practical use of large capable models is to compress them by reducing their parameter count, while retaining as much as possible of their performance.

One of the prevalent techniques for compressing models is knowledge distillation (Hinton et al., 2015). Distillation is the process of training a model– the student– to replicate the knowledge of another model– the teacher– on a specific set of tasks. Typically, the student has fewer parameters than the teacher and as such, distillation can improve task-specific performance while maintaining lower inference cost and memory footprint than the teacher.

The large model could be an ensemble of separately trained models or a single large model trained with a very strong regularizer such as dropout. Once the large model has been trained, we can then use a different kind of training, which we call "distillation" to transfer the knowledge from the large model to a small model that is more suitable for deployment. Strong dropout helps ensure that the model learns robust and generalizable features rather than overfitting the training data.

However, current KD methods[1] for auto-regressive sequence models suffer from distribution mismatch between output sequences seen during training and those generated by student during inference. 

> Why this train-inference mismatch occurs:

- Auto-regressive nature of language models: These models generate text one token at a time, using previously generated tokens as context for the next prediction.

- Training process: During training, the student model is typically exposed to complete, correct sequences from the training data or teacher-generated sequences. The model learns to predict the next token given the correct previous tokens.

- Inference (generation) process: At inference time, the model generates text from scratch or continues from a given prompt. As it generates, it uses its own previous outputs as context for subsequent tokens.

- The mismatch: During training, the model always sees "correct" or "expert-generated" contexts. During inference, the model sees its own generated context, which may contain errors or be less optimal than the training contexts. As generation progresses, these small differences can compound, leading to increasingly divergent contexts.

- Consequences: The partial sequences encountered during inference can be quite different from those seen in training. The model may not have been trained to handle or recover from its own errors or less-than-optimal generations.


To address this issue, the authors [On-Policy Distillation paper] introduce Generalized Knowledge Distillation (GKD). 

> Instead of solely relying on a fixed set of output sequences, GKD trains the student on its self-generated output sequences by leveraging feedback from the teacher. 

> GKD offers flexibility to employ alternative loss functions between the student and teacher, which may be useful when the student lacks the expressivity to mimic the teacher's distribution. 


![text](gkd_algo.png)

Teacher model: `Qwen2-7B-Instruct` 

Student model: `Qwen2-0.5B-Instruct`


**High level tasks:**

1. SFT student model on Teacher completions dataset. 

2. Use the SFT model to generate the output sequences on the fly with temperature of 1 to encourage diversity in generated sequences.

3. Obtain token level feedback from teacher's logits and leverage the GKDTrainer from Kashif's branch in TRL where we choose the divergence to optimize between teacher and student distributions

[1] Current KD methods for auto-regressive sequence models require, generating a fixed set of output sequences from the teacher model (Supervised KD) or a fixed dataset of sequences that the teacher can label by assigning token-level probabilities. 

References: 

- https://arxiv.org/abs/1503.02531 
- https://arxiv.org/pdf/2306.13649
- https://arxiv.org/abs/2106.05237
- https://pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html 


Plan is to start with something a bit smaller (to validate it works before scaling up):

- Distill Qwen2-7B-Instruct to Qwen2-0.5B
- Use LMSYS prompts as the source of generating student / teacher completions
- `GKDTrainer` branch from Kashif R.: https://github.com/huggingface/trl/pull/1814
- Dataset: https://huggingface.co/datasets/andito/chatbot_arena_completions [Produced using `Qwen2-7B-Instruct` completions]
- PR: https://github.com/huggingface/llm-swarm/pull/31/commits/f50230ca5a0cc880e6aab88127bb2dedae0368c7 
- PPO Trainer: https://huggingface.co/docs/trl/ppo_trainer 
- GKD Trainer example script: https://github.com/kashif/trl/blob/gkd-trainer/examples/scripts/gkd.py

In [36]:
# files = !ls

# if 'requirements.txt' in files:
#     !pip install -r requirements.txt

In [6]:
# !pip install pandas
# !pip install pyarrow
# !pip install huggingface_hub
# !pip install datasets
# !pip install transformers
# !pip install torch
# !pip install torchvision
#!pip install trl
# !pip install accelerate -U
#!pip install tensorboard
#!pip install bitsandbytes
#!pip install peft
#!pip install --upgrade trl
!pip install git+https://github.com/kashif/trl.git@

In [1]:
#!pip list

In [9]:
import os
os.getcwd()

'/data/kd_exps/trl/on_policy'

In [5]:
import pandas as pd
import random
from datetime import datetime
from datasets import load_dataset
from datasets import Dataset
from datasets import DatasetDict
import accelerate
import gc
import inspect
from typing import List, Dict, Any, Union

from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, BitsAndBytesConfig
import sympy
from huggingface_hub import HfFolder, Repository, create_repo, login

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

from utils import print_gpu_memory

from trl import SFTConfig, SFTTrainer#, GKDConfig, GKDTrainer

In [32]:
#import sys

# """
# reset kernel if imports do not work
# """

#Get the directory of the current script
#current_dir = os.path.dirname(os.path.abspath(__file__))

#Add the directory containing your .py file to sys.path
#sys.path.insert(0, '/data/kd_exps/trl')
#sys.path.insert(1, '/data/kd_exps/trl/trl')

#sys.path.append(os.path.abspath('/data/kd_exps/trl'))
#sys.path.append(os.path.abspath('/data/kd_exps/trl/trl'))

# for path in sys.path:
#     print(path)

In [29]:
#del sys.path[0]

In [33]:
for path in sys.path:
    print(path)

/home/user/miniconda/lib/python39.zip
/home/user/miniconda/lib/python3.9
/home/user/miniconda/lib/python3.9/lib-dynload


In [35]:
from trl import (
    GKDConfig,
    GKDTrainer,
    ModelConfig,
    RichProgressCallback
)

from trl.trainer.utils import pad

ImportError: cannot import name 'GKDConfig' from 'trl' (/home/user/miniconda/lib/python3.9/site-packages/trl/__init__.py)

In [82]:
class DataCollatorForLastCompletionLM(DataCollatorForLanguageModeling):
    """
    Data collator for language modeling that ignores all tokens except the last completion.
    It also separates prompts and completions for easy access.

    Args:
        tokenizer: The tokenizer to use for encoding the data.
        mlm (bool): Whether to use masked language modeling. Default is False.
        response_template (str): The template that marks the start of a response.
        ignore_index (int): The index to use for ignoring tokens in loss calculation.
    """

    def __init__(
        self, tokenizer, mlm: bool = False, response_template: str = "### Response:\n", ignore_index: int = -100
    ):
        super().__init__(tokenizer=tokenizer, mlm=mlm)
        self.response_template = self.tokenizer.encode(response_template, add_special_tokens=False)
        self.ignore_index = ignore_index

    def torch_call(self, examples: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch = super().torch_call(examples)

        prompts = []
        completions = []

        for i in range(len(examples)):
            input_ids = batch["input_ids"][i]
            labels = batch["labels"][i]

            # Find all occurrences of the response template
            response_starts = [
                j
                for j in range(len(input_ids) - len(self.response_template) + 1)
                if input_ids[j : j + len(self.response_template)].tolist() == self.response_template
            ]

            if not response_starts:
                # If no response template is found, treat the whole input as a prompt
                prompts.append(input_ids)
                completions.append(torch.tensor([]))
                labels[:] = self.ignore_index
            else:
                # Get the start of the last response
                last_response_start = response_starts[-1]

                # Separate prompt and completion
                prompts.append(input_ids[:last_response_start])
                completions.append(input_ids[last_response_start:])

                # Set labels for all tokens before the last response to ignore_index
                labels[:last_response_start] = self.ignore_index

        # Add prompts and completions to the batch
        batch["prompts"] = pad(prompts, padding_value=self.tokenizer.pad_token_id, padding_side="left")
        batch["completions"] = pad(completions, padding_value=self.tokenizer.pad_token_id, padding_side="right")

        return batch

In [83]:
# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

# Get the number of available GPUs
num_gpus = torch.cuda.device_count()
print(f"Number of available GPUs: {num_gpus}")

CUDA Available: True
Number of available GPUs: 1


In [84]:
# Set the device (replace 'cuda:0' with the appropriate GPU if you have multiple GPUs)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

In [85]:
print_gpu_memory()

Current GPU memory available: 42.30 GB
Current GPU memory allocated: 18.14 GB
Current GPU memory cached: 31.81 GB


### 1. Read Qwen2-7B-Instruct completions dataset

In [86]:
df = load_dataset("andito/chatbot_arena_completions")

In [87]:
df

DatasetDict({
    train: Dataset({
        features: ['question_id', 'messages'],
        num_rows: 32980
    })
})

In [88]:
df['train'][:1]

{'question_id': ['58210e39b3fd4441a2bd4a518bb44c2d'],
 'messages': [[{'content': 'What is the difference between OpenCL and CUDA?',
    'role': 'user'},
   {'content': "CUDA (Compute Unified Device Architecture) and OpenCL (Open Computing Language) are two popular GPU computing platforms developed by different companies.\n\nCUDA is a parallel computing platform and application programming interface (API) model created by NVIDIA. It was specifically developed for NVIDIA GPU hardware and is based on NVIDIA's architecture, making it highly optimized for specific NVIDIA GPUs. CUDA allows developers to directly access the processing power of the GPU and develop parallel-accelerated applications that can run at essentially the same speed as NVIDIA processors. It has a proprietary nature with exclusive use of NVIDIA GPUs and adherence to NVIDIA's hardware and software standards.\n\nOpenCL, on the other hand, is a platform-independent (not tied to a specific hardware vendor) open standard for 

In [89]:
# Split test / eval set
test_size = 1000 / len(df['train'])
test_size

0.030321406913280776

In [90]:
# Split the dataset
split_dataset = df['train'].train_test_split(test_size=test_size, seed=42)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['question_id', 'messages'],
        num_rows: 31980
    })
    test: Dataset({
        features: ['question_id', 'messages'],
        num_rows: 1000
    })
})

In [91]:
# Create a new DatasetDict with both splits
df_v1 = DatasetDict({
                     'train': split_dataset['train'],
                     'test': split_dataset['test']
                   })

df_v1

DatasetDict({
    train: Dataset({
        features: ['question_id', 'messages'],
        num_rows: 31980
    })
    test: Dataset({
        features: ['question_id', 'messages'],
        num_rows: 1000
    })
})

### 2. SFT student model on Teacher completions dataset

In [92]:
# Load model
#tokenizer_student = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
#model_student = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

In [93]:
# save to disk
#tokenizer_student.save_pretrained(os.getcwd() + "/model")
#model_student.save_pretrained(os.getcwd() + "/model")

In [94]:
# Load model from disk
tokenizer_student = AutoTokenizer.from_pretrained(os.getcwd() + "/model", device_map="cuda:0")
#model_student = AutoModelForCausalLM.from_pretrained(os.getcwd() + "/model", device_map="cuda:0")

In [95]:
# special tokens
#tokenizer_student.special_tokens_map

In [96]:
#model_student.device

In [97]:
#model_student

Let's add special tokens and format the conversation dataset

In [98]:
# To Pandas
pd_dict = {}

for split, data in df_v1.items():
    
    pd_dict[split] = df_v1[split].to_pandas()

In [99]:
f"Train DF shape: {pd_dict['train'].shape}"

'Train DF shape: (31980, 2)'

In [100]:
f"Test DF shape: {pd_dict['test'].shape}"

'Test DF shape: (1000, 2)'

In [101]:
# Function to format the conversation dataset
def format_conversation(sample):
    
    #print('sample', sample)
    
    text = ""
    for message in sample:  # Assuming single conversation per example
        
        if message['role'] == 'user':
            text = f"<|im_start|>### User: {message['content']}<|im_end|>"
        
        elif message['role'] == 'assistant':
            text += f"<|im_start|>### Assistant: {message['content']}"
            return text + tokenizer_student.eos_token

In [102]:
# Apply format function to the dataset
pd_dict['train']['texts'] = pd_dict['train'].apply(lambda x: format_conversation(x['messages']), axis=1)
pd_dict['test']['texts'] = pd_dict['test'].apply(lambda x: format_conversation(x['messages']), axis=1)

In [103]:
# Max Sequence Length
max_length_train = max(len(text) for text in pd_dict['train']['texts'])
max_length_test = max(len(text) for text in pd_dict['test']['texts'])

max_length = max(max_length_train, max_length_test) 

f"Max Length: {max_length}"

'Max Length: 11796'

In [104]:
# To HF Dataset
hf_dict = {}

for split, df in pd_dict.items():
            
    hf_df = Dataset.from_pandas(df)
        
    hf_dict[split] = hf_df
    
hf_dataset = DatasetDict(hf_dict)

#### 2.1 SFTTrainer

**Loss function: standard cross-entropy loss**[1]

This is the primary loss function used for language model training. It measures the difference between the predicted probability distribution of tokens and the actual distribution (Completions from Teacher model). 

> The loss is calculated token-by-token across the entire sequence.

> It's then averaged over all non-ignored tokens in the batch.

> The model predicts the next token given all the previous tokens, thus it is called an autoregressive model.


Loss for a single token prediction: `L = -Σ(y_i * log(p_i))`

Where:

    - `y_i` is the true probability of class i (usually 1 for the correct class, 0 for others)
    - `p_i` is the predicted probability of class i


[1] https://discuss.huggingface.co/t/fine-tune-with-sfttrainer/67311 


In [105]:
print_gpu_memory()

Current GPU memory available: 42.30 GB
Current GPU memory allocated: 18.14 GB
Current GPU memory cached: 31.81 GB


In [106]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fri Aug 30 22:17:36 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  | 00000000:20:1C.0 Off |                    0 |
| N/A   37C    P0              75W / 400W |  30853MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [107]:
# # Trainer

# instruction_template = "### User:"
# response_template = "### Assistant:"
# collator = DataCollatorForCompletionOnlyLM(instruction_template=instruction_template, response_template=response_template, tokenizer=tokenizer, mlm=False)

# sft_config = SFTConfig(
#                         max_seq_length=max_length,
#                         per_device_train_batch_size=5,
#                         per_device_eval_batch_size=5,
#                         gradient_accumulation_steps=2,
#                         learning_rate=0.0001,
#                         weight_decay=0.01, # L2 regularization
#                         lr_scheduler_type="cosine",
#                         warmup_ratio=0.1, # warm up for first 10% of steps
#                         num_train_epochs=3,
#                         logging_dir=f"./logs/{timestamp}/",
#                         report_to=["tensorboard"],
#                         logging_steps=500, 
#                         eval_strategy="steps",
#                         eval_steps=500,  # Evaluate steps
#                         save_strategy="steps",
#                         save_steps=5000, # save checkpoint
#                         output_dir = "./results",
#                         dataset_text_field="texts",
#                         #fp16=True,  # Enable mixed precision training
#                         max_grad_norm=1.0,
#                         gradient_checkpointing=True,  # Enable gradient checkpointing
#                         optim="adamw_torch_fused", 
#                         load_best_model_at_end=True,
#                         metric_for_best_model="eval_loss"
#                       )

# trainer = SFTTrainer(
#                         model,
#                         train_dataset=hf_dataset['train'],
#                         eval_dataset=hf_dataset['test'],
#                         args=sft_config,
#                         data_collator=collator,
#                         tokenizer=tokenizer
#                     )

In [108]:
#trainer.train()

In [109]:
# Evaluate the model on the test set after training
#test_results = trainer.evaluate()
#print("Final test results:", test_results)

In [110]:
# Save the best model
#trainer.save_model("./fine_tuned_model")

In [111]:
# Push to HF Hub / Repo

# Load the SFT model and tokenizer from disk
#model_sft = AutoModelForCausalLM.from_pretrained("./fine_tuned_model")
#tokenizer_sft = AutoTokenizer.from_pretrained("./fine_tuned_model")

# Set the organization and repository name
#org_name = "Distillation-Hugs"  # Replace with your organization name
#repo_name = "kd_exps"  # Replace with your desired model name

# Create a new repository
#repo_url = create_repo(repo_id="Distillation-Hugs/kd_exps", repo_type="model", private=False)

# Clone the empty repository
#repo = Repository(local_dir="./hf_model_repo", clone_from=repo_url)

In [112]:
#model.push_to_hub(repo_id=f"{org_name}/{repo_name}")

In [113]:
#tokenizer.push_to_hub(repo_id=f"{org_name}/{repo_name}")

#### 2.2 Evaluate 

https://www.philschmid.de/evaluate-llm-mixeval

https://github.com/philschmid/MixEval

In [114]:
# Set environment variables
#os.environ["MODEL_PARSER_API"] = "sk-proj-PKQLrbYQ1UkZUldUz7WXT3BlbkFJQYgrM7Jd1eiHzwaO11A0"

In [115]:
#os.getcwd()

In [116]:
# # Run the shell command
# !python -m mix_eval.evaluate \
#     --data_path hf://zeitgeist-ai/mixeval \
#     --model_path /data/kd_exps/trl/on_policy/model  \
#     --output_dir results/eval \
#     --model_name local_chat\
#     --benchmark mixeval_hard \
#     --version 2024-06-01 \
#     --batch_size 20 \
#     --api_parallel_num 20

In [ ]:
# Checkout SFT `Qwen0.5B` model from HF Hub
qwen_05b_tokenizer = AutoTokenizer.from_pretrained("Distillation-Hugs/kd_exps", device_map="cuda:0")
qwen_05_model = AutoModelForCausalLM.from_pretrained("Distillation-Hugs/kd_exps", device_map="cuda:0")

RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
qwen_05_model

In [ ]:
qwen_05_model.device

In [ ]:
print_gpu_memory()

In [ ]:
# Print GKDTrainer arguments for inspection
for name, param in inspect.signature(GKDTrainer).parameters.items():
    print(f"{param}")

In [ ]:
"""

KL:

Kullback-Leibler divergence, also known as relative entropy, is a measure of the difference between two probability distributions P and Q. 

Mathematically, for discrete probability distributions, it is defined as:

KL(P||Q) = Σ P(x) * log(P(x)/Q(x))

For continuous distributions, the sum is replaced by an integral.

Interpretation:

- KL divergence can be interpreted as the amount of information lost when Q is used to approximate P. 
- It's not symmetric: KL(P||Q) ≠ KL(Q||P).

----------------------------------------------------------------------------------------

JSD: 

The Jensen-Shannon Divergence is a method of measuring the similarity between two probability distributions. 
It is based on the Kullback-Leibler (KL) divergence but has some notable advantages:

It's symmetric: JSD(P||Q) = JSD(Q||P)
It always has a finite value
Its square root is a metric (satisfies the triangle inequality)

Mathematically, for two probability distributions P and Q, the JSD is defined as:
JSD(P||Q) = 1/2 * D(P||M) + 1/2 * D(Q||M)

Where:

D is the Kullback-Leibler divergence
M = 1/2(P + Q)


Note: JS divergence is sometimes preferred in scenarios where a symmetric measure is needed or when dealing with distributions that might have zero probabilities in some regions.

"""

gkd_config = GKDConfig(
    
                        #max_seq_length=max_length,
                        do_train=True,
                        per_device_train_batch_size=1,
                        gradient_accumulation_steps=4,
                        learning_rate=0.0001,
                        weight_decay=0.01, # L2 regularization
                        lr_scheduler_type="cosine",
                        warmup_ratio=0.1, # warm up for first 10% of steps
                        num_train_epochs=3,
                        logging_dir=f"./logs/{timestamp}/",
                        #report_to=["tensorboard"],
                        logging_steps=500, 
                        eval_strategy="steps",
                        eval_steps=500,  # Evaluate steps
                        save_strategy="steps",
                        save_steps=5000, # save checkpoint
    
                        fp16=True,  # Enable mixed precision training
                        max_grad_norm=1.0, 
                        gradient_checkpointing=True,  # Enable gradient checkpointing
                        optim="adamw_torch_fused", 
                        load_best_model_at_end=True,
                        metric_for_best_model="eval_loss",

                        output_dir="./gkd/",
                        temperature=1, # diversity in student generated output sequences
                        lmbda=0.5, # % of times we generate labels via model.generate vs labels/completions from the data
                        beta=0.5 # when beta=0 it should be the KL div and when beta=1 its the inverse and for others its the linear interpolation between them
                        #teacher_model_name_or_path=AutoModelForCausalLM.from_pretrained("./fine_tuned_model")

                      )

In [ ]:
qwen_05b_tokenizer.pad_token = qwen_05b_tokenizer.eos_token
qwen_05b_tokenizer.model_max_length = max_length

instruction_template = "### User:"
response_template = "### Assistant:"
collator = DataCollatorForLastCompletionLM(
                                            #instruction_template=instruction_template, 
                                            response_template=response_template, 
                                            tokenizer=qwen_05b_tokenizer, 
                                            mlm=False,
                                          )

In [ ]:
# Load Quantized Teacher model

# bnb_config = BitsAndBytesConfig(
#                                     load_in_8bit=True,
#                                     llm_int8_threshold=6.0,  # This is the default threshold for quantization
#                                     llm_int8_enable_fp32_cpu_offload=False,  # Offload non-quantized layers to CPU if necessary
#                                     llm_int8_has_fp16_weight=True # weights are stored in 16-bits and do not need to converted back and forth for training/FT 
#                                )

tokenizer_teacher = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B-Instruct")

model_teacher = AutoModelForCausalLM.from_pretrained(
                                                     "Qwen/Qwen2-7B-Instruct",
                                                     #quantization_config=bnb_config
                                                     torch_dtype=torch.bfloat16,
                                                     device_map="cuda:0"
                                                    )

model_teacher

In [ ]:
f"Student model vocab size: {qwen_05b_tokenizer.vocab_size}"

In [ ]:
f"Teacher model vocab size: {model_teacher.vocab_size}"

In [ ]:
student_vocab_dict = qwen_05b_tokenizer.get_vocab()
teacher_vocab_dict = tokenizer_teacher.get_vocab()

In [ ]:
non_overlap_student = {k:v for k,v in student_vocab_dict.items() if k not in teacher_vocab_dict.keys() or teacher_vocab_dict[k] != v}
non_overlap_student

In [ ]:
non_overlap_teacher = {k:v for k,v in teacher_vocab_dict.items() if k not in student_vocab_dict.keys() or student_vocab_dict[k] != v}
non_overlap_teacher

There are no non-overlapping key-value pairs between teacher and student. There are likely duplicate keys in teacher. Let's find and drop them. 

In [ ]:
"""
Iterate over k,v in teacher tokenizer. 

- If k not in seen Set, add k,v to unique_dict. 

- After adding to unique_dict, append to seen. 

- If k in seen Set, skip loop and do not add to unique_dict
"""

unique_dict = {}
seen = set()

for k,v in teacher_vocab_dict.items():
    
    if k not in seen:
        unique_dict[k] = v
        seen.add(v)
        
len(unique_dict)

151646

In [ ]:
tokenizer_teacher_v1 = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B-Instruct", vocab=unique_dict)
tokenizer_teacher_v1

Qwen2TokenizerFast(name_or_path='Qwen/Qwen2-7B-Instruct', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [37]:
print(f"Original Embedding dimension: {model_teacher.get_input_embeddings().num_embeddings}")
model_teacher.resize_token_embeddings(tokenizer_teacher_v1.vocab_size)
print(f"Resized Embedding dimension: {model_teacher.get_input_embeddings().num_embeddings}")

Original Embedding dimension: 152064
Resized Embedding dimension: 151643


In [38]:
f"Student model vocab size: {qwen_05b_tokenizer.vocab_size}"

'Student model vocab size: 151643'

In a **forward pass** through a transformer model like _Qwen2ForCausalLM_, several key steps occur. Here’s a breakdown of what typically happens during the forward pass:

1. Input Embeddings

- Token Embeddings: Input token IDs are passed through an embedding layer (embed_tokens). This layer converts token IDs into dense vectors of fixed size (embedding size).

- Positional Embeddings: Positional information is added to the token embeddings. This helps the model understand the position of each token in the sequence. In Qwen2ForCausalLM, rotary embeddings (rotary_emb) are used to incorporate positional information in a way that scales with sequence length.

2. Stack of Decoder Layers

The model contains multiple decoder layers. Each decoder layer consists of the following components:

- Self-Attention Mechanism:

Projection Layers: Queries (q_proj), keys (k_proj), and values (v_proj) are projected from the input embeddings or the hidden states from the previous layer.

Attention Calculation: The attention scores are computed using the scaled dot-product attention mechanism. This involves calculating the dot product of queries and keys, applying a softmax function to get attention weights, and using these weights to compute a weighted sum of the values.

Output Projection: The result of the attention mechanism is projected back to the original hidden size (o_proj).
In Qwen2ForCausalLM, Qwen2SdpaAttention handles these operations.

- Residual Connection and Layer Normalization:

Residual Connection: The output of the self-attention mechanism is added to the original input of the attention layer (residual connection).

Layer Normalization: The result is normalized using layer normalization (input_layernorm).
Feed-Forward Network (MLP):

Projection Layers: The hidden states are passed through a feed-forward network with multiple linear projections (gate_proj, up_proj, down_proj).

Activation Function: An activation function (SiLU) is applied to introduce non-linearity.

In Qwen2ForCausalLM, the feed-forward network is handled by Qwen2MLP.

Residual Connection: The output of the feed-forward network is added to the hidden states from the previous layer (residual connection).

Layer Normalization: The result is normalized using layer normalization (post_attention_layernorm).

3. Output Layer

Linear Transformation: The final hidden states are projected to the vocabulary size using a linear layer (lm_head). This produces logits for each token in the vocabulary, which can be used to compute probabilities and generate text.


In [53]:
print_gpu_memory()

Current GPU memory available: 42.30 GB
Current GPU memory allocated: 33.89 GB
Current GPU memory cached: 35.43 GB


In [39]:
# Data needs to be split into prompts and completions. 

gkd_trainer = GKDTrainer(
                            model=qwen_05_model,
                            teacher_model=model_teacher,
                            train_dataset=hf_dataset['train'],
                            eval_dataset=hf_dataset['test'],
                            args=gkd_config,
                            tokenizer=qwen_05b_tokenizer,
                            data_collator=collator
                        )  
                        

/data/kd_exps/trl/trl/trainer/sft_trainer.py:289: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/data/kd_exps/trl/trl/trainer/sft_trainer.py:366: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/31980 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/home/user/miniconda/lib/python3.9/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [40]:
print_gpu_memory()

Current GPU memory available: 42.30 GB
Current GPU memory allocated: 18.13 GB
Current GPU memory cached: 19.58 GB


In [41]:
torch.cuda.empty_cache()
gkd_trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/user/miniconda/lib/python3.9/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/user/miniconda/lib/python3.9/site-packages/torch/utils/checkpoint.py:92: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cach

Teacher Input IDs: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  94344,    279,   6672,   1948,
            279,  13440,    323,  21619,   5746,    304,  13142, 151645,    198,
         151644,  77091,    198,    641,  13142,     11,    279,  34243,    323,
          57682,   5746,    525,   1378,   2989,  35972,   5746,    429,    614,
           2155,   8357,    382,    785,  34243,    729,    320,  51255,   2075,
             11,    379,    593,    374,    264,  19259,  18927,   7982,    429,
          16555,    279,  18927,    315,   3709,    264,   2393,    304,    264,
           1378,  10188,    530,   7868,   9342,     11,   1741,    438,  64661,
            264,  16254,  10917,     13,   1084,    374,   4512,    438,   1447,
          51255,   2075,     11,    379,      8,    284,  84216,   2075,    488,
            379,      8,    608,  84216,   2075,      8,  84216,   7021,    692,
         

../aten/src/ATen/native/cuda/Indexing.cu:1284: indexSelectLargeIndex: block: [720,0,0], thread: [0,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1284: indexSelectLargeIndex: block: [720,0,0], thread: [1,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1284: indexSelectLargeIndex: block: [720,0,0], thread: [2,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1284: indexSelectLargeIndex: block: [720,0,0], thread: [3,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1284: indexSelectLargeIndex: block: [720,0,0], thread: [4,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1284: indexSelectLargeIndex: block: [720,0,0], thread: [5,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1284: indexSelectLargeIndex: block: [720,0,0], 

RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
